# Test Run Results Summary

Performance of 13 new models across 7 benchmarks (test runs with `--limit 2`).

In [1]:
import json
import os
import glob
import pandas as pd

TEST_DIR = "/home/jiaruil5/proxy_bench/proxy_bench_data/test"

# Models to include (api_model_id -> display name)
MODELS = {
    "azure_ai/Kimi-K2.5": "Kimi-K2.5",
    "azure_ai/Kimi-K2-Thinking": "Kimi-K2-Thinking",
    "azure_ai/DeepSeek-V3.2": "DeepSeek-V3.2-Reasoner",
    "azure/gpt-5.2": "GPT-5.2",
    "fireworks_ai/accounts/fireworks/models/minimax-m2p5": "MiniMax-M2.5",
    "fireworks_ai/accounts/fireworks/models/glm-4p7": "GLM-4.7",
    "fireworks_ai/accounts/fireworks/models/minimax-m2p1": "MiniMax-M2.1",
    "fireworks_ai/accounts/fireworks/models/glm-5": "GLM-5",
    "anthropic/claude-opus-4-6": "Claude Opus 4.6",
    "azure/gpt-5.2-codex": "GPT-5.2-Codex",
    "fireworks_ai/accounts/fireworks/models/qwen3-next-80b-a3b-thinking": "Qwen3-Next-80B-Thinking",
    "fireworks_ai/accounts/fireworks/models/qwen3-coder-30b-a3b-instruct": "Qwen3-Coder-30B",
    "nvidia/nvidia-nemotron-3-nano-30b-a3b-bf16": "Nemotron-3-Nano-30B",
}

# Map api_model_id to directory name (replace / with __)
def model_to_dir(model_id):
    return model_id.replace("/", "__")

# Benchmark directories and their primary metric
BENCHMARKS = {
    "ifeval": {
        "dir": "ifeval",
        "task": "ifeval",
        "metric": "prompt_level_strict_acc,none",
        "display": "IFEval\n(strict)",
    },
    "acp_gen_2shot": {
        "dir": "acp_gen_2shot",
        "task": "__avg__",  # average across subtasks
        "metric": "score,acp_grammar_parse",
        "display": "ACPBench\n(gen 2-shot)",
    },
    "mbpp_chat": {
        "dir": "mbpp_chat",
        "task": "mbpp_chat",
        "metric": "pass_at_1,extract_code",
        "display": "MBPP\n(chat)",
    },
    "humaneval_chat": {
        "dir": "humaneval_chat",
        "task": "humaneval_chat",
        "metric": "pass@1,create_test",
        "display": "HumanEval\n(chat)",
    },
    "gpqa_diamond": {
        "dir": "gpqa_diamond_cot_zeroshot,gpqa_main_cot_zeroshot,gpqa_extended_cot_zeroshot",
        "task": "gpqa_diamond_cot_zeroshot",
        "metric": "exact_match,flexible-extract",
        "display": "GPQA\n(diamond)",
    },
    "gpqa_main": {
        "dir": "gpqa_diamond_cot_zeroshot,gpqa_main_cot_zeroshot,gpqa_extended_cot_zeroshot",
        "task": "gpqa_main_cot_zeroshot",
        "metric": "exact_match,flexible-extract",
        "display": "GPQA\n(main)",
    },
    "gpqa_extended": {
        "dir": "gpqa_diamond_cot_zeroshot,gpqa_main_cot_zeroshot,gpqa_extended_cot_zeroshot",
        "task": "gpqa_extended_cot_zeroshot",
        "metric": "exact_match,flexible-extract",
        "display": "GPQA\n(extended)",
    },
    "aime25": {
        "dir": "aime25",
        "task": "aime25",
        "metric": "exact_match,none",
        "display": "AIME25",
    },
    "logiqa": {
        "dir": "logiqa_cot_zeroshot",
        "task": "logiqa_cot_zeroshot",
        "metric": "exact_match,flexible-extract",
        "display": "LogiQA\n(CoT)",
    },
}


def get_latest_result(benchmark_dir, model_dir):
    """Get the latest results JSON for a model on a benchmark."""
    path = os.path.join(TEST_DIR, benchmark_dir, model_dir)
    if not os.path.isdir(path):
        return None
    result_files = sorted(glob.glob(os.path.join(path, "results_*.json")))
    if not result_files:
        return None
    with open(result_files[-1]) as f:
        return json.load(f)


def extract_score(result_data, task, metric):
    """Extract a score from result data."""
    if result_data is None:
        return None
    results = result_data.get("results", {})
    if task == "__avg__":
        # Average the metric across all subtasks
        vals = []
        for t, d in results.items():
            if metric in d:
                vals.append(d[metric])
        return sum(vals) / len(vals) if vals else None
    if task in results and metric in results[task]:
        return results[task][metric]
    return None


# Build the table
rows = []
for model_id, display_name in MODELS.items():
    model_dir = model_to_dir(model_id)
    row = {"Model": display_name, "API Model ID": model_id}
    for bench_key, bench_info in BENCHMARKS.items():
        data = get_latest_result(bench_info["dir"], model_dir)
        score = extract_score(data, bench_info["task"], bench_info["metric"])
        row[bench_info["display"]] = score
    rows.append(row)

df = pd.DataFrame(rows)
df = df.set_index("Model")

In [2]:
# Format scores as percentages
score_cols = [c for c in df.columns if c != "API Model ID"]

def fmt(val):
    if val is None or pd.isna(val):
        return "-"
    return f"{val*100:.1f}"

df_display = df.copy()
for col in score_cols:
    df_display[col] = df_display[col].apply(fmt)

print("Test Results (--limit 2, scores in %)")
print("=" * 40)
df_display

Test Results (--limit 2, scores in %)


,API Model ID,IFEval\n(strict),ACPBench\n(gen 2-shot),MBPP\n(chat),HumanEval\n(chat),GPQA\n(diamond),GPQA\n(main),GPQA\n(extended),AIME25,LogiQA\n(CoT)
Model,,,,,,,,,,
Kimi-K2.5,azure_ai/Kimi-K2.5,100.0,68.8,100.0,100.0,100.0,100.0,100.0,50.0,100.0
Kimi-K2-Thinking,azure_ai/Kimi-K2-Thinking,100.0,0.0,100.0,100.0,100.0,50.0,50.0,-,100.0
DeepSeek-V3.2-Reasoner,azure_ai/DeepSeek-V3.2,100.0,31.2,100.0,100.0,50.0,100.0,50.0,50.0,100.0
GPT-5.2,azure/gpt-5.2,100.0,68.8,100.0,100.0,100.0,100.0,100.0,50.0,50.0
MiniMax-M2.5,fireworks_ai/accounts/fireworks/models/minimax...,100.0,81.2,50.0,100.0,100.0,100.0,50.0,100.0,50.0
GLM-4.7,fireworks_ai/accounts/fireworks/models/glm-4p7,100.0,0.0,100.0,100.0,100.0,100.0,100.0,50.0,0.0
MiniMax-M2.1,fireworks_ai/accounts/fireworks/models/minimax...,100.0,0.0,100.0,100.0,50.0,100.0,50.0,50.0,100.0
GLM-5,fireworks_ai/accounts/fireworks/models/glm-5,100.0,68.8,100.0,100.0,100.0,100.0,100.0,50.0,100.0
Claude Opus 4.6,anthropic/claude-opus-4-6,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


In [3]:
# Styled HTML table with color coding
df_scores = df[score_cols].copy()

def color_score(val):
    if val is None or pd.isna(val):
        return "background-color: #f0f0f0; color: #999"
    if val >= 0.8:
        return "background-color: #c6efce; color: #006100"
    elif val >= 0.5:
        return "background-color: #ffeb9c; color: #9c5700"
    else:
        return "background-color: #ffc7ce; color: #9c0006"

styled = (
    df_scores
    .style
    .map(color_score)
    .format(lambda v: f"{v*100:.1f}%" if pd.notna(v) else "-")
    .set_caption("Test Run Results (--limit 2)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center"), ("white-space", "pre-line")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ])
)
styled

,IFEval (strict),ACPBench (gen 2-shot),MBPP (chat),HumanEval (chat),GPQA (diamond),GPQA (main),GPQA (extended),AIME25,LogiQA (CoT)
Model,,,,,,,,,
Kimi-K2.5,100.0%,68.8%,100.0%,100.0%,100.0%,100.0%,100.0%,50.0%,100.0%
Kimi-K2-Thinking,100.0%,0.0%,100.0%,100.0%,100.0%,50.0%,50.0%,-,100.0%
DeepSeek-V3.2-Reasoner,100.0%,31.2%,100.0%,100.0%,50.0%,100.0%,50.0%,50.0%,100.0%
GPT-5.2,100.0%,68.8%,100.0%,100.0%,100.0%,100.0%,100.0%,50.0%,50.0%
MiniMax-M2.5,100.0%,81.2%,50.0%,100.0%,100.0%,100.0%,50.0%,100.0%,50.0%
GLM-4.7,100.0%,0.0%,100.0%,100.0%,100.0%,100.0%,100.0%,50.0%,0.0%
MiniMax-M2.1,100.0%,0.0%,100.0%,100.0%,50.0%,100.0%,50.0%,50.0%,100.0%
GLM-5,100.0%,68.8%,100.0%,100.0%,100.0%,100.0%,100.0%,50.0%,100.0%
Claude Opus 4.6,100.0%,0.0%,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
